# TCS Tracking Bot (Google Colab)

Checks TCS tracking numbers from your Excel sheet and writes the live status back into the same sheet.

**How to use:** run the cells from top to bottom with the ▶ button on the left of each cell.
Cell 1 sets up, cell 2 picks your Excel file, then use cell 3 (one number) or cell 4 (all rows), and cell 5 to download the updated file.

Close the Excel file on your computer before uploading it.

In [ ]:
#@title 1. Setup (run once per session)
import os, subprocess, sys
if not os.path.isdir("/content/tcs-tracking-bot"):
    subprocess.run(["git", "clone", "-q", "https://github.com/obaidkhan009/tcs-tracking-bot.git", "/content/tcs-tracking-bot"], check=True)
else:
    subprocess.run(["git", "-C", "/content/tcs-tracking-bot", "pull", "-q"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "/content/tcs-tracking-bot/requirements.txt"], check=True)
os.chdir("/content/tcs-tracking-bot")
EXCEL = None
print("Setup done.")

In [ ]:
#@title 2. Choose your Excel file
#@markdown Pick **upload** to upload the .xlsx from your computer, or **drive** to use a file saved in Google Drive (it is updated in place).
source = "upload" #@param ["upload", "drive"]
#@markdown Only for drive: path inside My Drive, for example `Shipments/shipments.xlsx`
drive_path = "shipments.xlsx" #@param {type:"string"}

import os
if source == "upload":
    from google.colab import files
    uploaded = files.upload()
    names = [n for n in uploaded if n.lower().endswith(".xlsx")]
    if not names:
        raise SystemExit("Please upload a .xlsx file.")
    EXCEL = os.path.abspath(names[0])
else:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    EXCEL = os.path.join("/content/drive/MyDrive", drive_path)
    if not os.path.exists(EXCEL):
        raise SystemExit(f"Not found in My Drive: {drive_path}")
print("Using:", EXCEL)

In [ ]:
#@title 3. Check one tracking number
tracking_number = "" #@param {type:"string"}
show_history = False #@param {type:"boolean"}

import subprocess, sys
if not tracking_number.strip():
    raise SystemExit("Type a tracking number in the box above, then run this cell.")
cmd = [sys.executable, "tracker.py"]
if EXCEL: cmd.append(EXCEL)
cmd += ["--track", tracking_number.strip()]
if show_history: cmd.append("--history")
subprocess.run(cmd, check=False)

In [ ]:
#@title 4. Refresh every row in the sheet
only_pending = True #@param {type:"boolean"}
#@markdown Tick "only_pending" to skip rows already marked DELIVERED or RETURNED.

import subprocess, sys
if not EXCEL:
    raise SystemExit("Run cell 2 first to choose the Excel file.")
cmd = [sys.executable, "tracker.py", EXCEL, "--all"]
if only_pending: cmd.append("--pending")
subprocess.run(cmd, check=False)

In [ ]:
#@title 5. Download the updated Excel file
#@markdown Needed only if you uploaded the file in cell 2. A Google Drive file is already updated in place.
if not EXCEL:
    raise SystemExit("Nothing to download yet. Run cell 2 first.")
if EXCEL.startswith("/content/drive/"):
    print("Your Google Drive file is already updated:", EXCEL)
else:
    from google.colab import files
    files.download(EXCEL)